# 🧠 AI Logic Engine: High-Precision Knowledge Ingestion
ဤ Notebook သည် ဉာဏ်ရည်တုကို အသုံးပြု၍ High-Quality Knowledge Triplets (Subject, Verb, Object) များ ထုတ်ယူပြီး Firestore သို့ စနစ်တကျ သွင်းပေးပါမည်။

### 🚀 Optimization Details:
- 🔄 **Resumable State:** `checkpoint.json` မှတစ်ဆင့် session ပြတ်သွားပါက ပြန်လည်စတင်နိုင်ပါသည်။
- 🛡️ **Normalization:** Subject နှင့် Object များကို lowercase/clean လုပ်ပေးခြင်းဖြင့် အချိတ်အဆက်မိစေပါသည်။
- 🧬 **Inheritance Support:** `is_a` verb ပါက `groups` field ထဲသို့ အလိုအလျောက် ထည့်သွင်းပေးပါသည်။
- ⚡ **Batching:** Firestore Batch writing (Limit: 450 operations) ကို အသုံးပြုပါသည်။

In [ ]:
# @title ⚡ Keep Alive Script
import IPython
js_code = """
function ClickConnect(){
  console.log("Clicking Connect...");
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}
setInterval(ClickConnect, 60000)
"""
display(IPython.display.Javascript(js_code))
print("✅ Keep-Alive script running.")

In [ ]:
# @title 🛠️ Step 1: Install Dependencies
!pip install -q transformers accelerate bitsandbytes firebase-admin tqdm
import os, json, re, torch, time, random
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
print("✅ Environment Setup Complete.")

In [ ]:
# @title 🔑 Step 2: Firebase Connection
if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'
db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Connected to Firestore: {DATABASE_ID}")

In [ ]:
# @title 🤖 Step 3: Load AI Model
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"⏳ Loading Model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")
model.eval()
print("✅ Model Ready.")

In [ ]:
# @title 🧬 Step 4: Logic Engine Configuration
CHECKPOINT_FILE = "checkpoint.json"
TARGET_GOAL = 3_000_000_000

CATEGORIES = [
    "General Science", "Biology", "Physics", "Chemistry", 
    "World History", "Ancient Civilizations", "Geography", 
    "Literature", "Philosophy", "Mathematics", "Technology",
    "Medicine", "Astronomy", "Economics", "Law"
]

def clean_id(text):
    # Lowercase and replace non-alphanumeric with underscore
    return re.sub(r'[^a-z0-9]', '_', text.lower().strip())

def load_state():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            try: return json.load(f)
            except: return {"count": 0, "cat_idx": 0}
    return {"count": 0, "cat_idx": 0}

def save_state(count, cat_idx):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"count": count, "cat_idx": cat_idx}, f)

def parse_triplets(text):
    triplets = []
    for line in text.strip().split('\n'):
        line = line.strip()
        if '|' in line:
            parts = line.split('|')
            if len(parts) == 3:
                s, v, o = parts[0].strip(), parts[1].strip(), parts[2].strip()
                if s and v and o:
                    triplets.append({
                        'subject': s, 
                        'verb': v.lower(), 
                        'object': o,
                        'sid': clean_id(s),
                        'oid': clean_id(o)
                    })
    return triplets

print("✅ Logic Engine Configured.")

In [ ]:
# @title 🚀 Step 5: Start Continuous Ingestion
state = load_state()
count = state['count']
cat_idx = state['cat_idx']

pbar = tqdm(initial=count, total=TARGET_GOAL, desc="Ingesting Triplets")

while count < TARGET_GOAL:
    try:
        current_cat = CATEGORIES[cat_idx % len(CATEGORIES)]
        prompt = f"Generate 15 factual knowledge triplets about {current_cat}. Format: Subject|Verb|Object. ONE per line. Use 'is_a' for categorization. NO intro text."
        
        messages = [
            {"role": "system", "content": "You are a symbolic logic data generator. Output ONLY triplets in format: Subject|Verb|Object."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            gen = model.generate(
                inputs.input_ids, 
                max_new_tokens=400, 
                do_sample=True, 
                temperature=0.8, 
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(gen[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        triplets = parse_triplets(response)
        
        if triplets:
            batch = db.batch()
            batch_size = 0
            
            for t in triplets:
                if not t['sid'] or not t['oid']: continue
                
                ref = db.collection('nodes').document(t['sid'])
                
                if t['verb'] == "is_a":
                    # Inheritance logic: Use cleaned OID so the engine can resolve the parent node
                    update_data = {'groups': firestore.ArrayUnion([t['oid']])}
                else:
                    # Standard Relation
                    update_data = {'relations': firestore.ArrayUnion([{
                        'verb': t['verb'], 
                        'targetId': t['oid'], 
                        'weight': 100
                    }])}
                
                # Set document with name for debugging/display
                update_data['name'] = t['subject']
                batch.set(ref, update_data, merge=True)
                batch_size += 1
                
                if batch_size >= 450: 
                    batch.commit()
                    batch = db.batch()
                    batch_size = 0
            
            if batch_size > 0: batch.commit()
            
            count += len(triplets)
            cat_idx += 1
            pbar.update(len(triplets))
            save_state(count, cat_idx)
        else:
            # Minimal delay if generation is empty
            time.sleep(0.5)
            
    except Exception as e:
        print(f"\n⚠️ Workflow Interruption: {e}. Cooling down for 5s...")
        time.sleep(5)

pbar.close()
print("✅ All Knowledge Ingested.")